In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
from math import log

st.set_page_config(
    page_title="ICU Cardiac Risk Dashboard (Demo)",
    layout="wide",
    page_icon="❤️",
)

def _red_green_red_cmap():
    return LinearSegmentedColormap.from_list("rgr", ["#d73027","#1a9850","#d73027"], N=256)

def gradient_state_bar(value, vmin, vmax, normal_low, normal_high, label, unit=""):
    fig, ax = plt.subplots(figsize=(6, 1.1))
    cmap = _red_green_red_cmap()
    grad = np.linspace(0, 1, 600).reshape(1, -1)
    ax.imshow(grad, aspect="auto", cmap=cmap, extent=[vmin, vmax, 0, 1])
    width = max(normal_high - normal_low, 1e-6)
    rect = Rectangle((normal_low, 0), width, 1, fill=False, lw=1.5, edgecolor="white", linestyle="--")
    ax.add_patch(rect)
    ax.axvline(value, color="black", lw=2)
    ax.set_yticks([])
    ax.set_xlim(vmin, vmax)
    ax.set_xlabel(f"{label}: {value:.2f} {unit}".strip())
    ax.tick_params(axis='x', labelsize=9)
    fig.tight_layout()
    return fig

INTERCEPT = -3.0

COEFS = {
    "age_z": 0.35,
    "sbp_low": 0.45, "sbp_high": 0.20,
    "dbp_low": 0.25,
    "hr_high": 0.30, "hr_low": 0.15,
    "rr_high": 0.25,
    "spo2_low": 0.40,
    "temp_high": 0.20,
    "lactate_high": 0.50,
    "troponin_log": 0.35,
    "creatinine_high": 0.40,
    "bun_high": 0.20,
    "wbc_high": 0.15,
    "hb_low": 0.15,
    "platelets_low": 0.20,
    "lvef_low": 0.30,
    "diabetes": 0.15, "htn": 0.05, "ckd": 0.30, "copd": 0.20,
    "smoker": 0.10, "prior_mi": 0.20,
    "inotropes": 0.60, "mechanical_vent": 0.70,
}

def engineer_features(inp: dict):
    feat = {}
    feat["age_z"] = (inp["age"] - 65.0) / 10.0
    feat["sbp_low"] = max(0.0, 100.0 - inp["sbp"]) / 10.0
    feat["sbp_high"] = max(0.0, inp["sbp"] - 140.0) / 10.0
    feat["dbp_low"] = max(0.0, 60.0 - inp["dbp"]) / 10.0
    feat["hr_high"] = max(0.0, inp["hr"] - 100.0) / 10.0
    feat["hr_low"] = max(0.0, 60.0 - inp["hr"]) / 10.0
    feat["rr_high"] = max(0.0, inp["rr"] - 20.0) / 5.0
    feat["spo2_low"] = max(0.0, 94.0 - inp["spo2"]) / 5.0
    feat["temp_high"] = max(0.0, inp["temp"] - 37.5) / 1.0
    feat["lactate_high"] = max(0.0, inp["lactate"] - 2.0)
    feat["troponin_log"] = max(0.0, np.log1p(inp["troponin"]) - np.log1p(35.0))
    feat["creatinine_high"] = max(0.0, inp["creatinine"] - 1.2)
    feat["bun_high"] = max(0.0, inp["bun"] - 20.0) / 10.0
    feat["wbc_high"] = max(0.0, inp["wbc"] - 12.0) / 2.0
    feat["hb_low"] = max(0.0, 13.0 - inp["hb"]) / 2.0
    feat["platelets_low"] = max(0.0, 150.0 - inp["platelets"]) / 50.0
    feat["lvef_low"] = max(0.0, 50.0 - inp["lvef"]) / 10.0
    for b in ["diabetes","htn","ckd","copd","smoker","prior_mi","inotropes","mechanical_vent"]:
        feat[b] = 1.0 if bool(inp[b]) else 0.0
    return feat

def predict_mortality(inp: dict):
    feat = engineer_features(inp)
    logit = INTERCEPT + sum(COEFS[k] * feat.get(k, 0.0) for k in COEFS.keys())
    prob = 1.0 / (1.0 + np.exp(-logit))
    contributions = {k: COEFS[k] * feat.get(k, 0.0) for k in COEFS.keys()}
    return prob, contributions, feat

def plot_contributions(contribs: dict, top_n=10):
    items = sorted(contribs.items(), key=lambda kv: abs(kv[1]), reverse=True)[:top_n]
    labels = [k for k, _ in items][::-1]
    vals = [v for _, v in items][::-1]
    fig, ax = plt.subplots(figsize=(6, 3.2))
    ax.barh(labels, vals)
    ax.axvline(0, lw=1, color="black")
    ax.set_xlabel("Contribution to log-odds (±)")
    fig.tight_layout()
    return fig

def section_header(title: str, subtitle: str = ""):
    st.markdown(f"## {title}")
    if subtitle:
        st.caption(subtitle)

def disclaimer():
    st.info("**Prototype only** — not for clinical use. The included model is a placeholder for demonstration.")

with st.sidebar.form("patient_form"):
    st.markdown("### Patient Details")
    c1, c2 = st.columns(2)
    with c1:
        age = st.number_input("Age (years)", min_value=18, max_value=100, value=72)
        sex = st.selectbox("Sex", ["Male","Female","Other"])
    with c2:
        inotropes = st.selectbox("On inotropes/vasopressors?", ["No","Yes"]) == "Yes"
        mechanical_vent = st.selectbox("On mechanical ventilation?", ["No","Yes"]) == "Yes"
    st.markdown("### Vitals")
    c3, c4 = st.columns(2)
    with c3:
        sbp = st.number_input("Systolic BP (mmHg)", min_value=60, max_value=220, value=92)
        dbp = st.number_input("Diastolic BP (mmHg)", min_value=30, max_value=140, value=55)
        hr = st.number_input("Heart Rate (bpm)", min_value=30, max_value=220, value=118)
        rr = st.number_input("Respiratory Rate (breaths/min)", min_value=6, max_value=60, value=24)
    with c4:
        spo2 = st.number_input("SpO₂ (%)", min_value=70, max_value=100, value=91)
        temp = st.number_input("Temperature (°C)", min_value=34.0, max_value=41.0, value=38.2, step=0.1)
        lvef = st.number_input("LVEF (%)", min_value=10, max_value=75, value=38)
    st.markdown("### Labs")
    c5, c6 = st.columns(2)
    with c5:
        troponin = st.number_input("Troponin (arbitrary units)", min_value=0.0, max_value=5000.0, value=450.0, step=1.0)
        lactate = st.number_input("Lactate (mmol/L)", min_value=0.0, max_value=20.0, value=3.8, step=0.1)
        creatinine = st.number_input("Creatinine (mg/dL)", min_value=0.2, max_value=15.0, value=2.1, step=0.1)
    with c6:
        bun = st.number_input("BUN (mg/dL)", min_value=2.0, max_value=150.0, value=36.0, step=0.5)
        wbc = st.number_input("WBC (K/µL)", min_value=1.0, max_value=50.0, value=14.0, step=0.1)
        hb = st.number_input("Hemoglobin (g/dL)", min_value=5.0, max_value=22.0, value=11.1, step=0.1)
        platelets = st.number_input("Platelets (K/µL)", min_value=10.0, max_value=1000.0, value=120.0, step=1.0)
    st.markdown("### Comorbidities")
    c7, c8, c9 = st.columns(3)
    with c7:
        diabetes = st.checkbox("Diabetes", value=True)
        htn = st.checkbox("Hypertension", value=True)
    with c8:
        ckd = st.checkbox("Chronic Kidney Disease", value=True)
        copd = st.checkbox("COPD", value=False)
    with c9:
        smoker = st.checkbox("Current/Recent Smoker", value=True)
        prior_mi = st.checkbox("Prior MI", value=True)
    submitted = st.form_submit_button("Run Risk Prediction")

patient = {
    "age": age, "sex": sex,
    "sbp": sbp, "dbp": dbp, "hr": hr, "rr": rr, "spo2": spo2, "temp": temp, "lvef": lvef,
    "troponin": troponin, "lactate": lactate, "creatinine": creatinine, "bun": bun,
    "wbc": wbc, "hb": hb, "platelets": platelets,
    "diabetes": diabetes, "htn": htn, "ckd": ckd, "copd": copd,
    "smoker": smoker, "prior_mi": prior_mi,
    "inotropes": inotropes, "mechanical_vent": mechanical_vent,
}

st.title("ICU Cardiac Risk Dashboard")
disclaimer()

left, right = st.columns([1.1, 1.2])

with left:
    section_header("Patient State", "Red–green–red bars show value vs. range; dashed box = normal.")
    ranges = [
        ("Systolic BP", patient["sbp"], 60, 200, 100, 120, "mmHg"),
        ("Diastolic BP", patient["dbp"], 30, 120, 60, 80, "mmHg"),
        ("Heart Rate", patient["hr"], 30, 200, 60, 100, "bpm"),
        ("Resp Rate", patient["rr"], 6, 40, 12, 20, "breaths/min"),
        ("SpO₂", patient["spo2"], 70, 100, 94, 100, "%"),
        ("Temperature", patient["temp"], 34, 41, 36.5, 37.5, "°C"),
        ("LVEF", patient["lvef"], 10, 75, 50, 70, "%"),
        ("Lactate", patient["lactate"], 0.0, 10.0, 0.5, 2.0, "mmol/L"),
        ("Creatinine", patient["creatinine"], 0.2, 10.0, 0.7, 1.3, "mg/dL"),
        ("Troponin", patient["troponin"], 0.0, 1000.0, 0.0, 35.0, "units"),
    ]
    for label, val, vmin, vmax, nlo, nhi, unit in ranges:
        fig = gradient_state_bar(val, vmin, vmax, nlo, nhi, label, unit)
        st.pyplot(fig, clear_figure=True)

with right:
    section_header("Risk Prediction", "Uses a demo logistic model — replace with your validated model.")
    if submitted:
        prob, contribs, feats = predict_mortality(patient)
        st.metric("Predicted In-ICU Mortality", f"{prob*100:.1f}%")
        st.caption("Interpretation: model probability, not a diagnosis.")
        with st.expander("Top Risk Factor Contributions (log-odds)", expanded=True):
            fig = plot_contributions(contribs, top_n=10)
            st.pyplot(fig, clear_figure=True)
        st.markdown("### What-if: Adjust Systolic BP")
        new_sbp = st.slider("Systolic BP (mmHg)", 60, 200, int(patient["sbp"]))
        if new_sbp != patient["sbp"]:
            alt = dict(patient)
            alt["sbp"] = new_sbp
            p2, _, _ = predict_mortality(alt)
            delta = (p2 - prob) * 100.0
            st.metric("Mortality if SBP adjusted", f"{p2*100:.1f}%", delta=f"{delta:+.1f} %pts")
        st.markdown("---")
        st.caption("Replace the demo `predict_mortality` function with your trained model and real feature pipeline.")
    else:
        st.info("Fill the form in the left sidebar and click **Run Risk Prediction**")

st.markdown("---")
st.caption("© Demo — do not use for clinical decision-making.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

def _rgr():
    return LinearSegmentedColormap.from_list("rgr", ["#d73027", "#1a9850", "#d73027"], N=256)

def draw_state_bar(ax, value, vmin, vmax, normal_low, normal_high, label, unit=""):
    grad = np.linspace(0, 1, 600).reshape(1, -1)
    ax.imshow(grad, aspect="auto", cmap=_rgr(), extent=[vmin, vmax, 0, 1])
    width = max(normal_high - normal_low, 1e-6)
    ax.add_patch(Rectangle((normal_low, 0), width, 1, fill=False,
                           lw=1.5, edgecolor="white", linestyle="--"))
    ax.axvline(float(value), color="black", lw=2)
    ax.set_xlim(vmin, vmax); ax.set_yticks([])
    ax.set_xlabel(f"{label}: {float(value):.2f} {unit}".strip(), fontsize=12)
    ax.tick_params(axis="x", labelsize=10)
    for s in ("top","right","left"): ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.2)

DEFAULT_BARS = [
    ("Systolic BP", "sbp", 60, 200, 100, 120, "mmHg"),
    ("Heart Rate", "hr", 40, 200, 60, 100, "bpm"),
    ("SpO₂", "spo2", 70, 100, 94, 100, "%"),
    ("Lactate", "lactate", 0.0, 10.0, 0.5, 2.0, "mmol/L"),
    ("Creatinine", "creatinine", 0.0, 10.0, 0.7, 1.3, "mg/dL"),
]

def plot_patient_state_bars(row: pd.Series, bars=DEFAULT_BARS, figsize=(6.4, 5.8)):
    use = [b for b in bars if b[1] in row and pd.notna(row[b[1]])]
    fig, axes = plt.subplots(len(use), 1, figsize=figsize)
    if len(use) == 1: axes = [axes]
    fig.subplots_adjust(hspace=0.6)
    for ax, (label, col, vmin, vmax, nlo, nhi, unit) in zip(axes, use):
        draw_state_bar(ax, row[col], vmin, vmax, nlo, nhi, label, unit)
    return fig

df = pd.read_csv("checkups.csv", parse_dates=["timestamp"], dayfirst=True, infer_datetime_format=True)

def latest_record_for_patient(df, patient_id):
    sub = df[df["patient_id"] == patient_id].sort_values("timestamp")
    if sub.empty:
        raise ValueError(f"No records for patient_id={patient_id}")
    return sub.iloc[-1]

patient_id = "A123"
row = latest_record_for_patient(df, patient_id)
fig = plot_patient_state_bars(row)
plt.show()
